In [39]:
#import statements
import pandas as pd
import matplotlib.pyplot as plt
import subprocess
import numpy as np
import nilearn.image
import nilearn.plotting
import tempfile
import copy
from torch.utils.data import random_split
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from pathlib import Path
import ants
import pydicom
import nibabel as nib
import os
from glob import glob
from tqdm import tqdm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torchinfo import summary
import nilearn

In [2]:
def dicom_to_adc_map(dicom_dir, output_path=None):
    """
    Convert DICOM files from a directory to a raw DTI ADC map (numpy array or NIfTI).
    
    Parameters:
    -----------
    dicom_dir : str or Path
        Path to directory containing DICOM files
    output_path : str or Path, optional
        If provided, saves as NIfTI file at this path
        
    Returns:
    --------
    adc_map : numpy.ndarray
        3D array of ADC values
    affine : numpy.ndarray
        Affine transformation matrix
    """
    dicom_dir = Path(dicom_dir)
    dicom_files = list(dicom_dir.glob("*.dcm")) + list(dicom_dir.glob("*.DCM"))
    
    if not dicom_files:
        raise ValueError(f"No DICOM files found in {dicom_dir}")
    
    # Read all DICOM files
    dicom_datasets = []
    for dcm_file in dicom_files:
        try:
            ds = pydicom.dcmread(str(dcm_file))
            dicom_datasets.append(ds)
        except Exception as e:
            print(f"Warning: Could not read {dcm_file}: {e}")
            continue
    
    if not dicom_datasets:
        raise ValueError("No valid DICOM files could be read")
    
    # Sort by slice location (ImagePositionPatient[2]) or InstanceNumber
    def get_slice_location(ds):
        if hasattr(ds, 'SliceLocation') and ds.SliceLocation is not None:
            return float(ds.SliceLocation)
        elif hasattr(ds, 'ImagePositionPatient') and ds.ImagePositionPatient is not None:
            return float(ds.ImagePositionPatient[2])
        elif hasattr(ds, 'InstanceNumber') and ds.InstanceNumber is not None:
            return float(ds.InstanceNumber)
        else:
            return 0.0
    
    dicom_datasets.sort(key=get_slice_location)
    
    # Get pixel data and spacing
    pixel_data = []
    for ds in dicom_datasets:
        # Get pixel array (apply rescale slope/intercept if present)
        pixels = ds.pixel_array.astype(np.float32)
        
        # Apply rescale slope and intercept if available
        if hasattr(ds, 'RescaleSlope'):
            pixels = pixels * float(ds.RescaleSlope)
        if hasattr(ds, 'RescaleIntercept'):
            pixels = pixels + float(ds.RescaleIntercept)
        
        pixel_data.append(pixels)
    
    # Stack into 3D array
    adc_map = np.stack(pixel_data, axis=0)  # Shape: (slices, height, width)
    
    # Get voxel spacing and orientation
    first_ds = dicom_datasets[0]
    
    # Pixel spacing
    if hasattr(first_ds, 'PixelSpacing') and first_ds.PixelSpacing:
        pixel_spacing = [float(x) for x in first_ds.PixelSpacing]
    else:
        pixel_spacing = [1.0, 1.0]  # Default
    
    # Slice thickness/spacing
    if hasattr(first_ds, 'SpacingBetweenSlices') and first_ds.SpacingBetweenSlices:
        slice_spacing = float(first_ds.SpacingBetweenSlices)
    elif len(dicom_datasets) > 1:
        # Calculate from slice locations
        slice_locs = [get_slice_location(ds) for ds in dicom_datasets]
        slice_spacing = abs(slice_locs[1] - slice_locs[0]) if len(slice_locs) > 1 else 1.0
    else:
        slice_spacing = 1.0
    
    # Create affine matrix (simple version - may need adjustment based on DICOM orientation)
    affine = np.eye(4)
    affine[0, 0] = pixel_spacing[0]  # x spacing
    affine[1, 1] = pixel_spacing[1]  # y spacing
    affine[2, 2] = slice_spacing     # z spacing
    
    # Get image position if available
    if hasattr(first_ds, 'ImagePositionPatient') and first_ds.ImagePositionPatient:
        affine[:3, 3] = [float(x) for x in first_ds.ImagePositionPatient]
    
    # Save as NIfTI if output path provided
    if output_path:
        nii_img = nib.Nifti1Image(adc_map, affine)
        nib.save(nii_img, str(output_path))
        print(f"Saved ADC map to {output_path}")
        print(f"Shape: {adc_map.shape}, Spacing: {pixel_spacing + [slice_spacing]}")
    
    return adc_map, affine

In [3]:
def dicom_to_t1_mri(dicom_dir, output_path=None):
    """
    Convert T1 MRI DICOM files from a directory to a NIfTI file.
    
    Parameters:
    -----------
    dicom_dir : str or Path
        Path to directory containing DICOM files
    output_path : str or Path, optional
        If provided, saves as NIfTI file at this path
        
    Returns:
    --------
    t1_volume : numpy.ndarray
        3D array of T1 MRI values
    affine : numpy.ndarray
        Affine transformation matrix
    """
    dicom_dir = Path(dicom_dir)
    dicom_files = list(dicom_dir.glob("*.dcm")) + list(dicom_dir.glob("*.DCM"))
    
    if not dicom_files:
        raise ValueError(f"No DICOM files found in {dicom_dir}")
    
    # Read all DICOM files
    dicom_datasets = []
    for dcm_file in dicom_files:
        try:
            ds = pydicom.dcmread(str(dcm_file))
            dicom_datasets.append(ds)
        except Exception as e:
            print(f"Warning: Could not read {dcm_file}: {e}")
            continue
    
    if not dicom_datasets:
        raise ValueError("No valid DICOM files could be read")
    
    # Sort by slice location (ImagePositionPatient[2]) or InstanceNumber
    def get_slice_location(ds):
        if hasattr(ds, 'SliceLocation') and ds.SliceLocation is not None:
            return float(ds.SliceLocation)
        elif hasattr(ds, 'ImagePositionPatient') and ds.ImagePositionPatient is not None:
            return float(ds.ImagePositionPatient[2])
        elif hasattr(ds, 'InstanceNumber') and ds.InstanceNumber is not None:
            return float(ds.InstanceNumber)
        else:
            return 0.0
    
    dicom_datasets.sort(key=get_slice_location)
    
    # Get pixel data and spacing
    pixel_data = []
    for ds in dicom_datasets:
        # Get pixel array (apply rescale slope/intercept if present)
        pixels = ds.pixel_array.astype(np.float32)
        
        # Apply rescale slope and intercept if available
        if hasattr(ds, 'RescaleSlope'):
            pixels = pixels * float(ds.RescaleSlope)
        if hasattr(ds, 'RescaleIntercept'):
            pixels = pixels + float(ds.RescaleIntercept)
        
        pixel_data.append(pixels)
    
    # Handle single 3D DICOM file or stack multiple 2D slices
    if len(pixel_data) == 1 and len(pixel_data[0].shape) == 3:
        # Single DICOM file with 3D volume
        t1_volume = pixel_data[0]
    elif len(pixel_data) > 0:
        # Multiple 2D slices - stack them
        t1_volume = np.stack(pixel_data, axis=0)  # Shape: (slices, height, width)
    else:
        raise ValueError("No valid pixel data found")
    
    # Get voxel spacing and orientation
    first_ds = dicom_datasets[0]
    
    # Pixel spacing
    if hasattr(first_ds, 'PixelSpacing') and first_ds.PixelSpacing:
        pixel_spacing = [float(x) for x in first_ds.PixelSpacing]
    else:
        pixel_spacing = [1.0, 1.0]  # Default
    
    # Slice thickness/spacing
    slice_spacing = 0.0
    if hasattr(first_ds, 'SpacingBetweenSlices') and first_ds.SpacingBetweenSlices:
        slice_spacing = float(first_ds.SpacingBetweenSlices)
    if slice_spacing == 0.0 and len(dicom_datasets) > 1:
        slice_locs = [get_slice_location(ds) for ds in dicom_datasets]
        if len(slice_locs) > 1:
            slice_spacing = abs(slice_locs[1] - slice_locs[0])
    if slice_spacing == 0.0 and hasattr(first_ds, 'SliceThickness') and first_ds.SliceThickness:
        slice_spacing = float(first_ds.SliceThickness)
    if slice_spacing == 0.0:
        if hasattr(first_ds, 'PerFrameFunctionalGroupsSequence'):
            pfg = first_ds.PerFrameFunctionalGroupsSequence
            if len(pfg) > 1:
                try:
                    pos0 = pfg[0].PlanePositionSequence[0].ImagePositionPatient
                    pos1 = pfg[1].PlanePositionSequence[0].ImagePositionPatient
                    slice_spacing = abs(float(pos1[2]) - float(pos0[2]))
                except (AttributeError, IndexError):
                    pass
    if slice_spacing == 0.0:
        slice_spacing = pixel_spacing[0]
    
    # Create affine matrix (simple version - may need adjustment based on DICOM orientation)
    affine = np.eye(4)
    affine[0, 0] = pixel_spacing[0]  # x spacing
    affine[1, 1] = pixel_spacing[1]  # y spacing
    affine[2, 2] = slice_spacing     # z spacing
    
    # Get image position if available
    if hasattr(first_ds, 'ImagePositionPatient') and first_ds.ImagePositionPatient:
        affine[:3, 3] = [float(x) for x in first_ds.ImagePositionPatient]
    
    # Save as NIfTI if output path provided
    if output_path:
        nii_img = nib.Nifti1Image(t1_volume, affine)
        nib.save(nii_img, str(output_path))
        print(f"Saved T1 MRI to {output_path}")
        print(f"Shape: {t1_volume.shape}, Spacing: {pixel_spacing + [slice_spacing]}")
    
    return t1_volume, affine

In [9]:
# Process all I* folders in t1_mri_dcm and save to t1_mri_raw
t1_base_dir = Path("model_data/adni/t1_long_data/t1_long_dcm/")
t1_output_dir = Path("model_data/adni/t1_long_data/t1_long_raw")

# Collect all I* folders that contain DICOM files
t1_i_folders = [f for f in t1_base_dir.rglob("I*") 
                if f.is_dir() and (any(f.glob("*.dcm")) or any(f.glob("*.DCM")))]

# Initialize counters
t1_processed = 0
t1_errors = 0

In [10]:
# Process each folder with progress bar
for i_folder in tqdm(t1_i_folders, desc="Processing T1 MRI DICOM folders"):
    try:
        # Extract image_id (the I* folder name, e.g., I1593283)
        image_id = i_folder.name
        
        # Extract subject_id from the relative path
        # The structure is: t1_mri_dcm/[subject_id]/.../I[image_id]
        relative_path = i_folder.relative_to(t1_base_dir)
        subject_id = relative_path.parts[0]  # First part is the subject_id
        
        # Create output filename: I[image_id]_[subject_id].nii
        output_filename = f"{image_id}_{subject_id}.nii"
        output_path = t1_output_dir / output_filename
        
        # Skip if already exists
        if output_path.exists():
            continue
        
        # Convert DICOM to T1 MRI NIfTI
        t1_volume, affine = dicom_to_t1_mri(i_folder, output_path)
        t1_processed += 1
        
    except Exception as e:
        print(f"\nError processing {i_folder}: {e}")
        t1_errors += 1
        continue

print(f"\nT1 MRI Processing complete!")
print(f"Successfully processed: {t1_processed}")
print(f"Errors: {t1_errors}")

Processing T1 MRI DICOM folders:   0%|          | 1/802 [00:00<07:16,  1.84it/s]

Saved T1 MRI to model_data/adni/t1_long_data/t1_long_raw/I1304390_003_S_6014.nii
Shape: (208, 256, 240), Spacing: [1.0, 1.0, 1.0]


Processing T1 MRI DICOM folders:   0%|          | 2/802 [00:00<05:25,  2.46it/s]

Saved T1 MRI to model_data/adni/t1_long_data/t1_long_raw/I965015_003_S_6014.nii
Shape: (208, 256, 240), Spacing: [1.0, 1.0, 1.0]


Processing T1 MRI DICOM folders:   0%|          | 3/802 [00:01<05:04,  2.62it/s]

...

Saved T1 MRI to model_data/adni/t1_long_data/t1_long_raw/I1123765_116_S_4855.nii
Shape: (176, 256, 240), Spacing: [1.0, 1.0, 1.0]


Processing T1 MRI DICOM folders: 100%|█████████▉| 801/802 [04:05<00:00,  3.28it/s]

Saved T1 MRI to model_data/adni/t1_long_data/t1_long_raw/I1544995_116_S_4855.nii
Shape: (176, 256, 240), Spacing: [1.0, 1.0, 1.0]


Processing T1 MRI DICOM folders: 100%|██████████| 802/802 [04:05<00:00,  3.26it/s]

Saved T1 MRI to model_data/adni/t1_long_data/t1_long_raw/I1304066_116_S_4855.nii
Shape: (176, 256, 240), Spacing: [1.0, 1.0, 1.0]

T1 MRI Processing complete!
Successfully processed: 802
Errors: 0


In [7]:
# Example usage for your ADNI data structure:
# For a single subject's ADC map:
adc_map, affine = dicom_to_adc_map(
    "model_data/adni/dti_dcm/002_S_1280/Axial_DTI_ADC/2017-03-13_14_12_25.0/I829303",
    output_path="model_data/adni/dti_raw/I829303_002_S_1280_ADC.nii"
)

Saved ADC map to model_data/adni/dti_raw/I829303_002_S_1280_ADC.nii
Shape: (80, 116, 116), Spacing: [2.5172414779663, 2.5172414779663, 2.0]


In [8]:
# Process all I* folders in dti_dcm and save to dti_raw
base_dir = Path("model_data/adni/dti_dcm")
output_dir = Path("model_data/adni/dti_raw")
output_dir.mkdir(parents=True, exist_ok=True)

# Find all folders matching pattern I* (starting with I followed by digits)
processed = 0
errors = 0

In [8]:
# Collect all I* folders that contain DICOM files
i_folders = [f for f in base_dir.rglob("I*") 
             if f.is_dir() and (any(f.glob("*.dcm")) or any(f.glob("*.DCM")))]

In [12]:
# Process each folder with progress bar
for i_folder in tqdm(i_folders, desc="Processing DICOM folders"):
    try:
        # Extract image_id (the I* folder name, e.g., I829303)
        image_id = i_folder.name
        
        # Extract subject_id from the relative path
        # The structure is: dti_dcm/[subject_id]/.../I[image_id]
        relative_path = i_folder.relative_to(base_dir)
        subject_id = relative_path.parts[0]  # First part is the subject_id
        
        # Create output filename: I[image_id]_[subject_id].nii
        output_filename = f"{image_id}_{subject_id}.nii"
        output_path = output_dir / output_filename
        
        # Skip if already exists
        if output_path.exists():
            continue
        
        # Convert DICOM to ADC map
        adc_map, affine = dicom_to_adc_map(i_folder, output_path)
        processed += 1
        
    except Exception as e:
        print(f"\nError processing {i_folder}: {e}")
        errors += 1
        continue

print(f"\nProcessing complete!")
print(f"Successfully processed: {processed}")
print(f"Errors: {errors}")

Processing DICOM folders:   1%|          | 2/299 [00:00<00:25, 11.54it/s]

Saved ADC map to model_data/adni/dti_raw/I1593306_023_S_6400.nii
Shape: (1, 81, 116, 116), Spacing: [1.0, 1.0, 2.0]
Saved ADC map to model_data/adni/dti_raw/I1193759_016_S_6789.nii
Shape: (80, 116, 116), Spacing: [2.0, 2.0, 2.0]


Processing DICOM folders:   1%|▏         | 4/299 [00:00<00:22, 12.89it/s]

Saved ADC map to model_data/adni/dti_raw/I1116506_137_S_4631.nii
Shape: (80, 116, 116), Spacing: [2.0, 2.0, 2.0]
Saved ADC map to model_data/adni/dti_raw/I1282878_137_S_4631.nii
Shape: (80, 116, 116), Spacing: [2.0, 2.0, 2.0]
Saved ADC map to model_data/adni/dti_raw/I1417012_137_S_4631.nii
Shape: (80, 116, 116), Spacing: [2.0, 2.0, 2.0]


Processing DICOM folders:   3%|▎         | 8/299 [00:00<00:21, 13.58it/s]

...

Saved ADC map to model_data/adni/dti_raw/I980392_041_S_6226.nii
Shape: (80, 116, 116), Spacing: [2.0, 2.0, 2.0]
Saved ADC map to model_data/adni/dti_raw/I1348003_041_S_6226.nii
Shape: (80, 116, 116), Spacing: [2.0, 2.0, 2.0]


Processing DICOM folders:  99%|█████████▉| 297/299 [00:24<00:00, 13.34it/s]

Saved ADC map to model_data/adni/dti_raw/I893558_941_S_4292.nii
Shape: (80, 116, 116), Spacing: [2.0, 2.0, 2.0]
Saved ADC map to model_data/adni/dti_raw/I1142378_022_S_6013.nii
Shape: (40, 180, 180), Spacing: [1.2222222089767, 1.2222222089767, 4.0]
Saved ADC map to model_data/adni/dti_raw/I1220335_094_S_6736.nii
Shape: (85, 116, 116), Spacing: [2.0, 2.0, 2.0]
Saved ADC map to model_data/adni/dti_raw/I532648_068_S_4340.nii
Shape: (40, 74, 74), Spacing: [3.0, 3.0, 3.0]


Processing DICOM folders: 100%|██████████| 299/299 [00:24<00:00, 12.15it/s]

Saved ADC map to model_data/adni/dti_raw/I926813_137_S_4482.nii
Shape: (80, 116, 116), Spacing: [2.0, 2.0, 2.0]
Saved ADC map to model_data/adni/dti_raw/I1246457_137_S_4482.nii
Shape: (80, 116, 116), Spacing: [2.0, 2.0, 2.0]

Processing complete!
Successfully processed: 299
Errors: 0


In [15]:
# Create a numpy array of file paths for files with shape (80, 116, 116)
dti_raw_dir = Path("model_data/adni/dti_raw")

# Get all .nii files (including .nii.gz)
nii_files = list(dti_raw_dir.glob("*.nii")) + list(dti_raw_dir.glob("*.nii.gz"))

# Filter files by shape (80, 116, 116)
valid_file_paths = []

for nii_file in tqdm(nii_files, desc="Checking file shapes"):
    try:
        # Load the NIfTI file
        nii_img = nib.load(str(nii_file))
        data = nii_img.get_fdata()
        
        # Check if shape is (80, 116, 116)
        if data.shape == (80, 116, 116):
            valid_file_paths.append(str(nii_file))
    except Exception as e:
        print(f"\nError loading {nii_file}: {e}")
        continue

# Convert to numpy array
valid_file_paths_array = np.array(valid_file_paths, dtype=object)

print(f"\nFound {len(valid_file_paths_array)} files with shape (80, 116, 116)")
print(f"Total files checked: {len(nii_files)}")

Checking file shapes: 100%|██████████| 299/299 [00:02<00:00, 130.89it/s]


Found 235 files with shape (80, 116, 116)
Total files checked: 299


In [14]:
# Create a numpy array of file paths for T1 MRI files with shape (176, 256, 240)
t1_mri_raw_dir = Path("model_data/adni/t1_mri_raw")

# Get all .nii files (including .nii.gz)
t1_nii_files = list(t1_mri_raw_dir.glob("*.nii")) + list(t1_mri_raw_dir.glob("*.nii.gz"))

# Filter files by shape (176, 256, 240)
t1_valid_file_paths = []

for nii_file in tqdm(t1_nii_files, desc="Checking T1 MRI file shapes"):
    try:
        # Load the NIfTI file
        nii_img = nib.load(str(nii_file))
        data = nii_img.get_fdata()
        
        # Check if shape is (176, 256, 240)
        if data.shape == (176, 256, 240):
            t1_valid_file_paths.append(str(nii_file))
    except Exception as e:
        print(f"\nError loading {nii_file}: {e}")
        continue

# Convert to numpy array
t1_valid_file_paths_array = np.array(t1_valid_file_paths, dtype=object)

print(f"\nFound {len(t1_valid_file_paths_array)} T1 MRI files with shape (176, 256, 240)")
print(f"Total T1 MRI files checked: {len(t1_nii_files)}")

Checking T1 MRI file shapes: 100%|██████████| 294/294 [00:18<00:00, 15.84it/s]


Found 236 T1 MRI files with shape (176, 256, 240)
Total T1 MRI files checked: 294


In [16]:
# Compare subjects in T1 MRI and DTI arrays to find the difference
import re

# Extract subject IDs from file paths
def extract_subject_id(path):
    """Extract subject ID from path like '.../I[image_id]_[subject_id].nii'"""
    filename = Path(path).name
    # Pattern: I[image_id]_[subject_id].nii
    match = re.search(r'I\d+_(.+?)\.nii', filename)
    if match:
        return match.group(1)
    return None

In [17]:
# Create filtered arrays with only files from common subjects
import re

# Extract subject ID from file path
def extract_subject_id(path):
    """Extract subject ID from path like '.../I[image_id]_[subject_id].nii'"""
    filename = Path(path).name
    match = re.search(r'I\d+_(.+?)\.nii', filename)
    if match:
        return match.group(1)
    return None

# Get common subjects (from previous cell output: 129 common subjects)
# We'll recalculate to be sure
t1_subjects = set()
for path in t1_valid_file_paths_array:
    subject_id = extract_subject_id(path)
    if subject_id:
        t1_subjects.add(subject_id)

dti_subjects = set()
for path in valid_file_paths_array:
    subject_id = extract_subject_id(path)
    if subject_id:
        dti_subjects.add(subject_id)

common_subjects = t1_subjects & dti_subjects

print(f"Common subjects: {len(common_subjects)}")

# Filter T1 MRI files to only include common subjects
t1_common_files = []
for path in t1_valid_file_paths_array:
    subject_id = extract_subject_id(path)
    if subject_id and subject_id in common_subjects:
        t1_common_files.append(path)

# Filter DTI files to only include common subjects
dti_common_files = []
for path in valid_file_paths_array:
    subject_id = extract_subject_id(path)
    if subject_id and subject_id in common_subjects:
        dti_common_files.append(path)

# Create numpy arrays
t1_common_array = np.array(t1_common_files, dtype=object)
dti_common_array = np.array(dti_common_files, dtype=object)

print(f"\nT1 MRI files with common subjects: {len(t1_common_array)}")
print(f"DTI files with common subjects: {len(dti_common_array)}")

Common subjects: 129

T1 MRI files with common subjects: 202
DTI files with common subjects: 213


In [24]:
# Extract subject IDs from T1 MRI files
t1_subjects = set()
for path in t1_valid_file_paths_array:
    subject_id = extract_subject_id(path)
    if subject_id:
        t1_subjects.add(subject_id)

# Extract subject IDs from DTI files
dti_subjects = set()
for path in valid_file_paths_array:
    subject_id = extract_subject_id(path)
    if subject_id:
        dti_subjects.add(subject_id)

# Find differences
t1_only = t1_subjects - dti_subjects
dti_only = dti_subjects - t1_subjects
common_subjects = t1_subjects & dti_subjects

print(f"T1 MRI files: {len(t1_valid_file_paths_array)}")
print(f"DTI files: {len(valid_file_paths_array)}")
print()
print(f"Unique subjects in T1 MRI: {len(t1_subjects)}")
print(f"Unique subjects in DTI: {len(dti_subjects)}")
print(f"Common subjects: {len(common_subjects)}")

T1 MRI files: 236
DTI files: 235

Unique subjects in T1 MRI: 154
Unique subjects in DTI: 146
Common subjects: 129


In [25]:
if t1_only:
    print(f"Subjects in T1 MRI but NOT in DTI ({len(t1_only)}):")
    for subject in sorted(t1_only):
        # Find the T1 file for this subject
        t1_files = [p for p in t1_valid_file_paths_array if subject in Path(p).name]
        print(f"  {subject}: {t1_files[0] if t1_files else 'No file found'}")

if dti_only:
    print(f"\nSubjects in DTI but NOT in T1 MRI ({len(dti_only)}):")
    for subject in sorted(dti_only):
        # Find the DTI file for this subject
        dti_files = [p for p in valid_file_paths_array if subject in Path(p).name]
        print(f"  {subject}: {dti_files[0] if dti_files else 'No file found'}")

if not t1_only and not dti_only:
    print("All subjects match! The difference must be in multiple files per subject.")
    
    # Check for subjects with multiple files
    from collections import Counter
    
    t1_subject_counts = Counter([extract_subject_id(p) for p in t1_valid_file_paths_array if extract_subject_id(p)])
    dti_subject_counts = Counter([extract_subject_id(p) for p in valid_file_paths_array if extract_subject_id(p)])
    
    # Find subjects with different file counts
    print("\nChecking for subjects with different file counts:")
    all_subjects = t1_subject_counts.keys() | dti_subject_counts.keys()
    differences = []
    
    for subject in sorted(all_subjects):
        t1_count = t1_subject_counts.get(subject, 0)
        dti_count = dti_subject_counts.get(subject, 0)
        if t1_count != dti_count:
            differences.append((subject, t1_count, dti_count))
            print(f"  {subject}: T1={t1_count} files, DTI={dti_count} files")
            
            # Show the actual files
            t1_files = [p for p in t1_valid_file_paths_array if subject in Path(p).name]
            dti_files = [p for p in valid_file_paths_array if subject in Path(p).name]
            print(f"    T1 files: {[Path(p).name for p in t1_files]}")
            print(f"    DTI files: {[Path(p).name for p in dti_files]}")
    
    if not differences:
        print("  No differences found in subject file counts.")

Subjects in T1 MRI but NOT in DTI (25):
  003_S_4288: model_data/adni/t1_mri_raw/I1498579_003_S_4288.nii
  016_S_6708: model_data/adni/t1_mri_raw/I1151022_016_S_6708.nii
  022_S_6013: model_data/adni/t1_mri_raw/I1142379_022_S_6013.nii
  036_S_2380: model_data/adni/t1_mri_raw/I1023430_036_S_2380.nii
  036_S_4491: model_data/adni/t1_mri_raw/I984343_036_S_4491.nii
  036_S_6179: model_data/adni/t1_mri_raw/I957103_036_S_6179.nii
  036_S_6189: model_data/adni/t1_mri_raw/I958275_036_S_6189.nii
  036_S_6231: model_data/adni/t1_mri_raw/I973656_036_S_6231.nii
  036_S_6316: model_data/adni/t1_mri_raw/I991861_036_S_6316.nii
  036_S_6466: model_data/adni/t1_mri_raw/I1059028_036_S_6466.nii
  073_S_4777: model_data/adni/t1_mri_raw/I351121_073_S_4777.nii
  094_S_6736: model_data/adni/t1_mri_raw/I1220298_094_S_6736.nii
  114_S_2392: model_data/adni/t1_mri_raw/I1399029_114_S_2392.nii
  114_S_6057: model_data/adni/t1_mri_raw/I1581805_114_S_6057.nii
  114_S_6063: model_data/adni/t1_mri_raw/I1574100_114_S_

In [20]:
# Create dataframes for DTI and T1 MRI with file info including dates
import re
from datetime import datetime

def extract_info_from_filename(filename):
    """Extract image_id and subject_id from filename like 'I[image_id]_[subject_id].nii'"""
    match = re.match(r'I(\d+)_(.+?)\.nii', filename)
    if match:
        return match.group(1), match.group(2)
    return None, None

def find_date_from_dicom_folder(image_id, subject_id, base_dir):
    """Find the date from the original DICOM folder structure"""
    # Search for the I[image_id] folder for this subject
    subject_dir = base_dir / subject_id
    if not subject_dir.exists():
        return None
    
    # Search recursively for the I[image_id] folder
    for i_folder in subject_dir.rglob(f"I{image_id}"):
        if i_folder.is_dir():
            # Get the parent directory which should contain the date
            date_folder = i_folder.parent
            date_str = date_folder.name
            # Date format: 2017-03-13_14_12_25.0 or similar
            try:
                # Extract just the date part (before the time)
                date_part = date_str.split('_')[0]
                # Try to parse it
                datetime.strptime(date_part, '%Y-%m-%d')
                return date_str
            except:
                # If parsing fails, return the folder name anyway
                return date_str
    return None

In [21]:
# Process DTI files
dti_base_dir = Path("model_data/adni/dti_dcm")
dti_raw_dir = Path("model_data/adni/dti_raw")
dti_files = list(dti_raw_dir.glob("*.nii")) + list(dti_raw_dir.glob("*.nii.gz"))

dti_data = []
for nii_file in tqdm(dti_files, desc="Processing DTI files"):
    filename = nii_file.name
    image_id, subject_id = extract_info_from_filename(filename)
    
    if image_id and subject_id:
        # Find date from original DICOM folder
        date_taken = find_date_from_dicom_folder(image_id, subject_id, dti_base_dir)
        
        dti_data.append({
            'file_name': filename,
            'file_path': str(nii_file),
            'image_id': image_id,
            'subject_id': subject_id,
            'date_taken': date_taken
        })

dti_df = pd.DataFrame(dti_data)
print(f"DTI DataFrame: {len(dti_df)} files")
print(f"\nDTI DataFrame columns: {dti_df.columns.tolist()}")
print(f"\nFirst few rows:")
dti_df.head()

Processing DTI files: 100%|██████████| 299/299 [00:00<00:00, 2741.63it/s]

DTI DataFrame: 299 files

DTI DataFrame columns: ['file_name', 'file_path', 'image_id', 'subject_id', 'date_taken']

First few rows:


,file_name,file_path,image_id,subject_id,date_taken
0,I1223070_012_S_4188.nii,model_data/adni/dti_raw/I1223070_012_S_4188.nii,1223070,012_S_4188,2019-09-04_09_54_08.0
1,I893053_941_S_6068.nii,model_data/adni/dti_raw/I893053_941_S_6068.nii,893053,941_S_6068,2017-08-21_18_31_11.0
2,I980392_041_S_6226.nii,model_data/adni/dti_raw/I980392_041_S_6226.nii,980392,041_S_6226,2018-03-29_13_21_31.0
3,I1043642_137_S_6557.nii,model_data/adni/dti_raw/I1043642_137_S_6557.nii,1043642,137_S_6557,2018-08-28_17_13_57.0
4,I879765_041_S_4143.nii,model_data/adni/dti_raw/I879765_041_S_4143.nii,879765,041_S_4143,2017-07-27_13_27_20.0


In [22]:
# Process T1 MRI files
t1_base_dir = Path("model_data/adni/t1_mri_dcm")
t1_raw_dir = Path("model_data/adni/t1_mri_raw")
t1_files = list(t1_raw_dir.glob("*.nii")) + list(t1_raw_dir.glob("*.nii.gz"))

t1_data = []
for nii_file in tqdm(t1_files, desc="Processing T1 MRI files"):
    filename = nii_file.name
    image_id, subject_id = extract_info_from_filename(filename)
    
    if image_id and subject_id:
        # Find date from original DICOM folder
        date_taken = find_date_from_dicom_folder(image_id, subject_id, t1_base_dir)
        
        t1_data.append({
            'file_name': filename,
            'file_path': str(nii_file),
            'image_id': image_id,
            'subject_id': subject_id,
            'date_taken': date_taken
        })

t1_df = pd.DataFrame(t1_data)
print(f"\n\nT1 MRI DataFrame: {len(t1_df)} files")
print(f"\nT1 MRI DataFrame columns: {t1_df.columns.tolist()}")
print(f"\nFirst few rows:")
t1_df.head()

Processing T1 MRI files: 100%|██████████| 294/294 [00:00<00:00, 2700.54it/s]



T1 MRI DataFrame: 294 files

T1 MRI DataFrame columns: ['file_name', 'file_path', 'image_id', 'subject_id', 'date_taken']

First few rows:


,file_name,file_path,image_id,subject_id,date_taken
0,I1413030_016_S_5057.nii,model_data/adni/t1_mri_raw/I1413030_016_S_5057...,1413030,016_S_5057,2021-02-19_13_16_36.0
1,I1493847_016_S_7002.nii,model_data/adni/t1_mri_raw/I1493847_016_S_7002...,1493847,016_S_7002,2021-09-14_15_36_50.0
2,I904007_024_S_5290.nii,model_data/adni/t1_mri_raw/I904007_024_S_5290.nii,904007,024_S_5290,2017-09-12_08_00_50.0
3,I916492_036_S_6088.nii,model_data/adni/t1_mri_raw/I916492_036_S_6088.nii,916492,036_S_6088,2017-10-12_10_18_44.0
4,I1634890_016_S_7014.nii,model_data/adni/t1_mri_raw/I1634890_016_S_7014...,1634890,016_S_7014,2021-11-30_12_04_08.0


In [25]:
# Create paired dataset for common subjects
# Match DTI and T1 MRI images by date similarity, maximizing pairs

from datetime import datetime
from collections import defaultdict

def parse_date(date_str):
    """Parse date string like '2017-03-13_14_12_25.0' to datetime"""
    if pd.isna(date_str) or date_str is None:
        return None
    try:
        # Extract date part (before first underscore after date)
        date_part = str(date_str).split('_')[0]
        return datetime.strptime(date_part, '%Y-%m-%d')
    except:
        return None

def date_difference(date1, date2):
    """Calculate absolute difference in days between two dates"""
    if date1 is None or date2 is None:
        return float('inf')  # Large penalty for missing dates
    return abs((date1 - date2).days)

def match_images_by_date(dti_images, t1_images):
    """
    Match DTI and T1 images by date similarity using greedy algorithm.
    Returns list of pairs (dti_idx, t1_idx) maximizing matches.
    """
    if len(dti_images) == 0 or len(t1_images) == 0:
        return []
    
    # Parse dates for all images
    dti_dates = [parse_date(d['date_taken']) for d in dti_images]
    t1_dates = [parse_date(t['date_taken']) for t in t1_images]
    
    # Create list of all possible pairs with their date differences
    pairs = []
    for i, dti_img in enumerate(dti_images):
        for j, t1_img in enumerate(t1_images):
            diff = date_difference(dti_dates[i], t1_dates[j])
            pairs.append((diff, i, j))
    
    # Sort by date difference (smallest first)
    pairs.sort(key=lambda x: x[0])
    
    # Greedy matching: take pairs with smallest date differences
    # ensuring each image is used only once
    matched_dti = set()
    matched_t1 = set()
    matches = []
    
    for diff, dti_idx, t1_idx in pairs:
        if dti_idx not in matched_dti and t1_idx not in matched_t1:
            matches.append((dti_idx, t1_idx))
            matched_dti.add(dti_idx)
            matched_t1.add(t1_idx)
    
    return matches

In [26]:
# Filter dataframes to only common subjects
common_subjects_set = set(common_subjects)  # From previous cell

dti_common = dti_df[dti_df['subject_id'].isin(common_subjects_set)].copy()
t1_common = t1_df[t1_df['subject_id'].isin(common_subjects_set)].copy()

print(f"DTI files for common subjects: {len(dti_common)}")
print(f"T1 MRI files for common subjects: {len(t1_common)}")

DTI files for common subjects: 217
T1 MRI files for common subjects: 212


In [40]:
# Diagnostic: Verify pairing logic and check for duplicate usage
print("=== DIAGNOSTIC ANALYSIS ===\n")

# Verify counts
print("1. File Counts:")
print(f"   DTI files for common subjects: {len(dti_common)}")
print(f"   T1 MRI files for common subjects: {len(t1_common)}")
print(f"   Total pairs created: {len(paired_df)}")
print(f"   Maximum possible pairs (min of both): {min(len(dti_common), len(t1_common))}")
print()

# Check if any DTI images are used multiple times
dti_paths_in_pairs = paired_df['dti_file_path'].tolist()
dti_duplicates = [path for path in dti_paths_in_pairs if dti_paths_in_pairs.count(path) > 1]

print("2. Duplicate DTI Usage:")
if dti_duplicates:
    print(f"   WARNING: {len(set(dti_duplicates))} unique DTI files used multiple times!")
    for dup_path in set(dti_duplicates):
        count = dti_paths_in_pairs.count(dup_path)
        print(f"   {Path(dup_path).name}: used {count} times")
else:
    print("   ✓ No DTI files used multiple times")
print()

# Check if any T1 images are used multiple times
t1_paths_in_pairs = paired_df['t1_file_path'].tolist()
t1_duplicates = [path for path in t1_paths_in_pairs if t1_paths_in_pairs.count(path) > 1]

print("3. Duplicate T1 Usage:")
if t1_duplicates:
    print(f"   WARNING: {len(set(t1_duplicates))} unique T1 files used multiple times!")
    for dup_path in set(t1_duplicates):
        count = t1_paths_in_pairs.count(dup_path)
        print(f"   {Path(dup_path).name}: used {count} times")
else:
    print("   ✓ No T1 files used multiple times")
print()

# Check unique files used
unique_dti_used = len(set(dti_paths_in_pairs))
unique_t1_used = len(set(t1_paths_in_pairs))

print("4. Unique Files Used:")
print(f"   Unique DTI files in pairs: {unique_dti_used} (out of {len(dti_common)} total)")
print(f"   Unique T1 files in pairs: {unique_t1_used} (out of {len(t1_common)} total)")
print()

# Check subjects with multiple pairs
print("5. Subjects with Multiple Pairs:")
subject_pair_counts = paired_df['subject_id'].value_counts()
multi_pair_subjects = subject_pair_counts[subject_pair_counts > 1]
print(f"   Subjects with >1 pair: {len(multi_pair_subjects)}")
if len(multi_pair_subjects) > 0:
    print(f"   Max pairs per subject: {subject_pair_counts.max()}")
    print(f"   Example subjects with multiple pairs:")
    for subject, count in multi_pair_subjects.head(5).items():
        print(f"     {subject}: {count} pairs")
        # Show the pairs for this subject
        subject_pairs = paired_df[paired_df['subject_id'] == subject]
        for idx, row in subject_pairs.iterrows():
            print(f"       Pair: DTI={Path(row['dti_file_path']).name} (date: {row['dti_date_taken']}) "
                  f"<-> T1={Path(row['t1_file_path']).name} (date: {row['t1_date_taken']})")
print()

# Verify the matching logic by checking a specific subject
print("6. Detailed Check for One Subject:")
example_subject = subject_pair_counts.index[0] if len(subject_pair_counts) > 0 else None
if example_subject:
    print(f"   Subject: {example_subject}")
    subject_dti = [d for d in dti_by_subject[example_subject]]
    subject_t1 = [t for t in t1_by_subject[example_subject]]
    print(f"   DTI images for this subject: {len(subject_dti)}")
    print(f"   T1 images for this subject: {len(subject_t1)}")
    print(f"   Pairs created for this subject: {subject_pair_counts[example_subject]}")
    
    # Show all DTI and T1 images
    print(f"   DTI images:")
    for d in subject_dti:
        print(f"     {d['file_name']} (date: {d['date_taken']})")
    print(f"   T1 images:")
    for t in subject_t1:
        print(f"     {t['file_name']} (date: {t['date_taken']})")
    
    # Show which ones were paired
    subject_pairs = paired_df[paired_df['subject_id'] == example_subject]
    print(f"   Paired:")
    for idx, row in subject_pairs.iterrows():
        print(f"     {Path(row['dti_file_path']).name} <-> {Path(row['t1_file_path']).name}")

print("\n=== END DIAGNOSTIC ===")

=== DIAGNOSTIC ANALYSIS ===

1. File Counts:
   DTI files for common subjects: 217
   T1 MRI files for common subjects: 212
   Total pairs created: 212
   Maximum possible pairs (min of both): 212

2. Duplicate DTI Usage:
   ✓ No DTI files used multiple times

3. Duplicate T1 Usage:
   ✓ No T1 files used multiple times

4. Unique Files Used:
   Unique DTI files in pairs: 212 (out of 217 total)
   Unique T1 files in pairs: 212 (out of 212 total)

5. Subjects with Multiple Pairs:
   Subjects with >1 pair: 51
   Max pairs per subject: 4
   Example subjects with multiple pairs:
     041_S_5100: 4 pairs
       Pair: DTI=I1555302_041_S_5100.nii (date: 2022-02-21_14_37_11.0) <-> T1=I1555295_041_S_5100.nii (date: 2022-02-21_14_02_53.0)
       Pair: DTI=I1185885_041_S_5100.nii (date: 2019-07-11_12_44_59.0) <-> T1=I1185877_041_S_5100.nii (date: 2019-07-11_12_06_22.0)
       Pair: DTI=I1033373_041_S_5100.nii (date: 2018-08-08_14_11_21.0) <-> T1=I1033364_041_S_5100.nii (date: 2018-08-08_13_42_32.0

In [29]:
# Group by subject
dti_by_subject = defaultdict(list)
t1_by_subject = defaultdict(list)

for _, row in dti_common.iterrows():
    dti_by_subject[row['subject_id']].append({
        'file_path': row['file_path'],
        'file_name': row['file_name'],
        'image_id': row['image_id'],
        'date_taken': row['date_taken']
    })

for _, row in t1_common.iterrows():
    t1_by_subject[row['subject_id']].append({
        'file_path': row['file_path'],
        'file_name': row['file_name'],
        'image_id': row['image_id'],
        'date_taken': row['date_taken']
    })

In [30]:
# Create pairs for each subject
paired_data = []

for subject_id in tqdm(sorted(common_subjects_set), desc="Matching images by date"):
    dti_images = dti_by_subject[subject_id]
    t1_images = t1_by_subject[subject_id]
    
    if len(dti_images) == 0 or len(t1_images) == 0:
        continue
    
    # Match images by date similarity
    matches = match_images_by_date(dti_images, t1_images)
    
    # Create pairs
    for dti_idx, t1_idx in matches:
        dti_img = dti_images[dti_idx]
        t1_img = t1_images[t1_idx]
        
        paired_data.append({
            'subject_id': subject_id,
            'dti_file_path': dti_img['file_path'],
            'dti_file_name': dti_img['file_name'],
            'dti_image_id': dti_img['image_id'],
            'dti_date_taken': dti_img['date_taken'],
            't1_file_path': t1_img['file_path'],
            't1_file_name': t1_img['file_name'],
            't1_image_id': t1_img['image_id'],
            't1_date_taken': t1_img['date_taken']
        })

Matching images by date: 100%|██████████| 129/129 [00:00<00:00, 68978.23it/s]


In [31]:
# Create dataframe
paired_df = pd.DataFrame(paired_data)

print(f"\n\nPaired Dataset Summary:")
print(f"Total pairs created: {len(paired_df)}")
print(f"Unique subjects with pairs: {paired_df['subject_id'].nunique()}")
print(f"\nPairs per subject distribution:")
print(paired_df['subject_id'].value_counts().value_counts().sort_index())



Paired Dataset Summary:
Total pairs created: 212
Unique subjects with pairs: 129

Pairs per subject distribution:
count
1    78
2    28
3    14
4     9
Name: count, dtype: int64


In [50]:
# Add "I" prefix if not already present
paired_df['dti_image_id'] = paired_df['dti_image_id'].apply(lambda x: f"I{x}" if not str(x).startswith('I') else x)
paired_df['t1_image_id'] = paired_df['t1_image_id'].apply(lambda x: f"I{x}" if not str(x).startswith('I') else x)

In [5]:
# Add group labels from CSV to paired_df
# Read the labels CSV file
labels_df = pd.read_csv("/Users/william.wakefield/PycharmProjects/ADRU/model_data/adni/t1MRI_labels_matched.csv")

# Create a mapping from Image Data ID to Group
# The Image Data ID column contains the T1 MRI image IDs
image_id_to_group = dict(zip(labels_df['Image Data ID'], labels_df['Group']))

print(f"\nImage ID to Group mapping created: {len(image_id_to_group)} entries")


Image ID to Group mapping created: 295 entries


In [55]:
# Match T1 image IDs from paired_df to get groups
# paired_df has 't1_image_id' column
paired_df['group'] = paired_df['t1_image_id'].map(image_id_to_group)

In [67]:
# Add label column to paired_df
# SMC and CN -> 0, everything else -> 1
paired_df['label'] = paired_df['group'].apply(lambda x: 0 if x in ['CN'] else 1)

In [15]:
paired_df_sep = paired_df[paired_df['group'].isin(['CN', 'AD'])].copy()
paired_df_sep['label_sep'] = paired_df_sep['group'].map({'CN': 0, 'AD': 1})

In [64]:
pd.DataFrame.to_csv(paired_df, "model_data/adni/paired_df_full.csv")

In [2]:
paired_df = pd.read_csv("model_data/adni/paired_df_full.csv", index_col=0)

In [ ]:
paired_df['dti_rotated_file_path'] = paired_df['dti_file_path'].str.replace('dti_raw', 'dti_rotated', regex=False)
paired_df['t1_rotated_file_path'] = paired_df['t1_file_path'].str.replace('t1_mri_raw', 't1_rotated', regex=False)

In [6]:
# Setup directories for registered and masked images
dti_reg_dir = Path("model_data/adni/dti_registered")
dti_masked_dir = Path("model_data/adni/dti_masked")
t1_reg_dir = Path("model_data/adni/t1_registered")
t1_masked_dir = Path("model_data/adni/t1_masked")

In [53]:
fixed = ants.image_read('model_data/mni_t1.nii')

In [40]:
wm_mask = nilearn.datasets.load_mni152_wm_mask()
gm_mask = nilearn.datasets.load_mni152_gm_mask()

In [52]:
in_dir = "model_data/adni/t1_mri_raw"
out_dir = "model_data/adni/t1_rotated"

def rot_x90_y180_z180(a):
    if a.ndim == 4:
        a = np.transpose(a, (0, 2, 1, 3))  # (X, Z, Y, T) - 90° on X
        a = np.flip(a, axis=2)              # -Y (completes 90° on X)
        a = np.flip(a, axis=0)              # -X (180° on Y)
        a = np.flip(a, axis=0)              # -X (180° on Z)
        a = np.flip(a, axis=1)              # -Z (180° on Z)
    else:
        a = np.transpose(a, (0, 2, 1))      # (X, Z, Y) - 90° on X
        a = np.flip(a, axis=2)              # -Y (completes 90° on X)
        a = np.flip(a, axis=0)              # -X (180° on Y)
        a = np.flip(a, axis=0)              # -X (180° on Z)
        a = np.flip(a, axis=1)              # -Z (180° on Z)
    return a

for fname in tqdm(f for f in os.listdir(in_dir) if f.endswith((".nii"))):
    nii = nib.load(os.path.join(in_dir, fname))
    data = rot_x90_y180_z180(nii.get_fdata())
    nib.save(
        nib.Nifti1Image(data, nii.affine, nii.header),
        os.path.join(out_dir, fname)
    )

294it [00:30,  9.71it/s]


In [60]:
t1_registered = []

for idx, row in tqdm(paired_df.iterrows(), total=len(paired_df),
    desc="Registering T1 to MNI"):
    t1_in_path = Path(row["t1_rotated_file_path"])

    t1_out_path = t1_reg_dir / t1_in_path.name

    # 1) Register subject T1 to MNI T1
    moving_t1 = ants.image_read(str(t1_in_path))
    reg_t1_to_mni = ants.registration(fixed=fixed, moving=moving_t1, type_of_transform="SyN")

    # Save T1 in MNI space
    ants.image_write(reg_t1_to_mni["warpedmovout"], str(t1_out_path))
    t1_registered.append(str(t1_out_path))

Registering T1 to MNI: 100%|██████████| 212/212 [41:58<00:00, 11.88s/it]   


In [59]:
dti_registered = []

for idx, row in tqdm(paired_df.iterrows(), total=len(paired_df),
    desc="Registering ADC to MNI"):
    dti_in_path = Path(row["dti_rotated_file_path"])
    dti_out_path = dti_reg_dir / dti_in_path.name

    # 2) Register subject ADC (DTI) to MNI t1 space
    moving_adc = ants.image_read(str(dti_in_path))
    reg_adc_to_mni = ants.registration(fixed=fixed, moving=moving_adc, type_of_transform="SyN")
    
    # Save ADC in MNI space
    ants.image_write(reg_adc_to_mni["warpedmovout"], str(dti_out_path))
    dti_registered.append(str(dti_out_path))

Registering ADC to MNI: 100%|██████████| 212/212 [09:47<00:00,  2.77s/it]


####New 802 pairs data

In [6]:
raw_dir = Path("model_data/adni/t1_long_data/t1_long_raw")
reg_dir = Path("model_data/adni/t1_long_data/t1_long_registered")
fixed = ants.image_read("model_data/mni_t1.nii")

def rot_x90_yneg180(a):
    """Apply +90° rotation around X axis, then -180° around Y axis."""
    a = np.transpose(a, (0, 2, 1))
    a = np.flip(a, axis=1)
    a = np.flip(a, axis=0)
    a = np.flip(a, axis=2)
    return a

nii_files = sorted(f for f in os.listdir(raw_dir) if f.endswith(".nii"))

In [28]:
for fname in tqdm(nii_files, desc="Orient + Register T1 to MNI"):
    out_path = reg_dir / fname

    nii = nib.load(str(raw_dir / fname))
    data = rot_x90_yneg180(nii.get_fdata())
    oriented_nii = nib.Nifti1Image(data, nii.affine, nii.header)

    with tempfile.NamedTemporaryFile(suffix=".nii", delete=False) as tmp:
        tmp_path = tmp.name
    nib.save(oriented_nii, tmp_path)

    moving = ants.image_read(tmp_path)
    reg = ants.registration(fixed=fixed, moving=moving, type_of_transform="SyN")
    ants.image_write(reg["warpedmovout"], str(out_path))

    os.unlink(tmp_path)

Orient + Register T1 to MNI: 100%|██████████| 802/802 [49:54<00:00,  3.73s/it]


In [23]:
FSLDIR = os.environ.get("FSLDIR", os.path.expanduser("~/fsl"))
FSL_BIN = os.path.join(FSLDIR, "share", "fsl", "bin")


def _fsl(tool, *args):
    """Run an FSL command, raising on failure."""
    cmd = [os.path.join(FSL_BIN, tool)] + [str(a) for a in args]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"{tool} failed ({result.returncode}): {result.stderr or result.stdout}")
    return result


def dti_dicom_to_adc(dicom_dir, output_path):
    """
    Convert raw DTI DICOM to a masked ADC (mean diffusivity) map using:
      dcm2niix → bet (brain mask) → dtifit → MD output.
    """
    dicom_dir = Path(dicom_dir)
    output_path = Path(output_path)

    with tempfile.TemporaryDirectory() as tmp_dir:
        tmp = Path(tmp_dir)
        dwi_base = str(tmp / "dwi")

        # 1) dcm2niix → 4D NIfTI + bval + bvec
        subprocess.run(
            ["dcm2niix", "-z", "y", "-f", "dwi", "-o", str(tmp), str(dicom_dir)],
            capture_output=True, text=True, check=True,
        )

        dwi_nii = tmp / "dwi.nii.gz"
        bval = tmp / "dwi.bval"
        bvec = tmp / "dwi.bvec"

        if not dwi_nii.exists():
            candidates = sorted(tmp.glob("*.nii.gz"))
            if not candidates:
                raise ValueError(f"dcm2niix produced no .nii.gz for {dicom_dir}")
            dwi_nii = candidates[0]
            stem = dwi_nii.stem.replace(".nii", "")
            bval = tmp / f"{stem}.bval"
            bvec = tmp / f"{stem}.bvec"

        for f in [dwi_nii, bval, bvec]:
            if not f.exists():
                raise FileNotFoundError(f"Missing expected file: {f}")

        # 2) bet — brain extraction
        brain_base = str(tmp / "dwi_brain")
        _fsl("bet", dwi_nii, brain_base, "-m", "-f", "0.3")
        mask = tmp / "dwi_brain_mask.nii.gz"

        # 3) dtifit — tensor fit, produces *_MD.nii.gz (= ADC)
        dti_base = str(tmp / "dti")
        _fsl("dtifit",
             "-k", dwi_nii,
             "-o", dti_base,
             "-m", mask,
             "-r", bvec,
             "-b", bval)

        md_nii = tmp / "dti_MD.nii.gz"
        if not md_nii.exists():
            raise FileNotFoundError(f"dtifit did not produce MD map for {dicom_dir}")

        # 5) Copy MD → output as uncompressed .nii
        img = nib.load(md_nii)
        nib.save(nib.Nifti1Image(img.get_fdata().astype(np.float32), img.affine, img.header),
                 str(output_path))

In [24]:
dti_base_dir = Path("model_data/adni/dti_long_data/dti_long_dcm")
dti_output_dir = Path("model_data/adni/dti_long_data/dti_long_raw")

In [25]:
dti_i_folders = sorted(
    [
        f
        for f in dti_base_dir.rglob("I*")
        if f.is_dir() and (any(f.glob("*.dcm")) or any(f.glob("*.DCM")))
    ],
    key=lambda p: str(p),
)

In [26]:
# Test on first DTI scan only
test_folder = dti_i_folders[0]
image_id = test_folder.name
subject_id = test_folder.relative_to(dti_base_dir).parts[0]
test_out = dti_output_dir / f"{image_id}_{subject_id}_TEST_MD.nii"

print(f"Input:  {test_folder}")
print(f"Output: {test_out}")

dti_dicom_to_adc(test_folder, test_out)

md = nib.load(test_out)
data = md.get_fdata()
print(f"MD shape: {data.shape}")
print(f"MD min={float(np.nanmin(data)):.6e}, max={float(np.nanmax(data)):.6e}")

Input:  model_data/adni/dti_long_data/dti_long_dcm/002_S_0413/Axial_DTI/2017-06-21_13_48_54.0/I863064
Output: model_data/adni/dti_long_data/dti_long_raw/I863064_002_S_0413_TEST_MD.nii
MD shape: (116, 116, 80)
MD min=-3.804509e-03, max=4.047291e-03


In [27]:
# Batch: all I* folders → dti_long_raw (skip if output already exists)
dti_errors = 0
for i_folder in tqdm(dti_i_folders, desc="DTI DICOM → ADC"):
    image_id = i_folder.name
    subject_id = i_folder.relative_to(dti_base_dir).parts[0]
    output_path = dti_output_dir / f"{image_id}_{subject_id}.nii"
    if output_path.exists():
        continue
    try:
        dti_dicom_to_adc(i_folder, output_path)
    except Exception as e:
        print(f"\nError processing {i_folder}: {e}")
        dti_errors += 1

print(f"\nDone. Errors: {dti_errors}")

DTI DICOM → ADC:  74%|███████▍  | 595/802 [1:43:10<1:21:00, 23.48s/it]


Error processing model_data/adni/dti_long_data/dti_long_dcm/128_S_0205/Axial_DTI/2022-08-04_10_30_38.0/I1612816: Missing expected file: /var/folders/4r/2w_vg8k91mldqsvxyw1y1cnml0bjbd/T/tmp8l_erus0/dwi.bval


DTI DICOM → ADC: 100%|██████████| 802/802 [2:28:14<00:00, 11.09s/it]  


Done. Errors: 1


In [35]:
dti_raw_dir = Path("model_data/adni/dti_long_data/dti_long_raw")
dti_reg_dir = Path("model_data/adni/dti_long_data/dti_long_registered")
t1_reg_dir = Path("model_data/adni/t1_long_data/t1_long_registered")

dti_nii_files = sorted(f for f in os.listdir(dti_raw_dir) if f.endswith(".nii"))

In [36]:
# Build lookup: subject_id → list of registered T1 filenames
t1_by_subject = {}
for t1f in os.listdir(t1_reg_dir):
    if not t1f.endswith(".nii"):
        continue
    sid = t1f.split("_", 1)[1].replace(".nii", "")
    t1_by_subject.setdefault(sid, []).append(t1f)

In [38]:
dti_reg_errors = 0
dti_reg_skipped = 0

for fname in tqdm(dti_nii_files, desc="Register DTI ADC to subject T1 (MNI)"):
    out_path = dti_reg_dir / fname
    if out_path.exists():
        continue

    subject_id = fname.split("_", 1)[1].replace(".nii", "")
    t1_candidates = t1_by_subject.get(subject_id)
    if not t1_candidates:
        dti_reg_skipped += 1
        continue

    try:
        fixed_t1 = ants.image_read(str(t1_reg_dir / t1_candidates[0]))
        moving_dti = ants.image_read(str(dti_raw_dir / fname))
        reg = ants.registration(fixed=fixed_t1, moving=moving_dti, type_of_transform="Rigid")
        ants.image_write(reg["warpedmovout"], str(out_path))
    except Exception as e:
        print(f"\nError registering {fname}: {e}")
        dti_reg_errors += 1

print(f"\nDone. Errors: {dti_reg_errors}, Skipped (no matching T1): {dti_reg_skipped}")

Register DTI ADC to subject T1 (MNI):  16%|█▌        | 130/801 [01:34<07:57,  1.40it/s]Exception Object caught: 

itk::ExceptionObject (0x3119ef8e0)
Location: "unknown" 
File: /Users/runner/work/ANTsPy/ANTsPy/itksource/Modules/Filtering/ImageStatistics/include/itkImageMomentsCalculator.hxx
Line: 124
Description: ITK ERROR: ImageMomentsCalculator(0x31ab15c00): Compute(): Total Mass of the image was zero. Aborting here to prevent division by zero later on.





Error registering I1092244_114_S_6039.nii: Registration failed with error code 1


Register DTI ADC to subject T1 (MNI):  28%|██▊       | 223/801 [02:38<06:41,  1.44it/s]Exception Object caught: 

itk::ExceptionObject (0x12a1f0d40)
Location: "unknown" 
File: /Users/runner/work/ANTsPy/ANTsPy/itksource/Modules/Filtering/ImageStatistics/include/itkImageMomentsCalculator.hxx
Line: 124
Description: ITK ERROR: ImageMomentsCalculator(0x31ab15c00): Compute(): Total Mass of the image was zero. Aborting here to prevent division by zero later on.





Error registering I1224877_020_S_6513.nii: Registration failed with error code 1


Register DTI ADC to subject T1 (MNI):  31%|███       | 245/801 [02:53<06:08,  1.51it/s]Exception Object caught: 

itk::ExceptionObject (0x3119ef8e0)
Location: "unknown" 
File: /Users/runner/work/ANTsPy/ANTsPy/itksource/Modules/Filtering/ImageStatistics/include/itkImageMomentsCalculator.hxx
Line: 124
Description: ITK ERROR: ImageMomentsCalculator(0x31028cfe0): Compute(): Total Mass of the image was zero. Aborting here to prevent division by zero later on.




...


Error registering I879213_114_S_6039.nii: Registration failed with error code 1


Register DTI ADC to subject T1 (MNI):  83%|████████▎ | 661/801 [07:50<01:47,  1.30it/s]Exception Object caught: 

itk::ExceptionObject (0x3102a2130)
Location: "unknown" 
File: /Users/runner/work/ANTsPy/ANTsPy/itksource/Modules/Filtering/ImageStatistics/include/itkImageMomentsCalculator.hxx
Line: 124
Description: ITK ERROR: ImageMomentsCalculator(0x31ab23cf0): Compute(): Total Mass of the image was zero. Aborting here to prevent division by zero later on.





Error registering I923015_114_S_6057.nii: Registration failed with error code 1


Register DTI ADC to subject T1 (MNI): 100%|██████████| 801/801 [09:34<00:00,  1.39it/s]


Done. Errors: 14, Skipped (no matching T1): 0
